# Skyscraper Race — playable notebook

**Download this one file** and open it in Jupyter Lab / Jupyter Notebook / VS Code / Google Colab.

No extra game files needed. Only Python 3 (stdlib). Optional: `ipywidgets` for buttons.

### How to play here
1. **Runtime → Run all** (or run cells top to bottom once).
2. Jump to **Play** and run the play cell.
3. Type commands when asked: `r` roll · `p` place on top · `s` scrap · `i N` insert at slot N · `q` quit

First tower to **150 m** wins. Strong materials at the bottom. Junk on top — or scrap it.

### The 6 DSA ideas in this notebook
| # | Idea | Where |
|---|------|--------|
| 1 | Hash table (dict) | material lookup |
| 2 | Linked list | tower floors |
| 3 | Recursion | weight above a floor |
| 4 | Stack | collapse |
| 5 | Min-heap | next challenge height |
| 6 | Binary search | floor at a height |


## 1. Settings
Tune these if you want a shorter game for class demos.


In [ ]:
TARGET_HEIGHT = 150
ROLLS_PER_TURN = 3
CAPACITY_CONSTANT = 1200

NUM_CHALLENGES = 3
CHALLENGE_MIN_HEIGHT = 50
CHALLENGE_MAX_HEIGHT = 130
CHALLENGE_MIN_GAP = 25

EARTHQUAKE_DAMAGE = 0.06
WINDSTORM_DAMAGE = 0.10
DRUNK_DRIVER_DAMAGE = 0.15

UNITS_PER_AMOUNT = 150

TIER_RANGES = {
    1: "junk", 2: "junk", 3: "junk",
    4: "common", 5: "common", 6: "common", 7: "common",
    8: "strong", 9: "strong",
}

print("Settings loaded. Win at", TARGET_HEIGHT, "m")


## 2. Hash table — materials

Why a dict? One key → one material. Easy to add rows. No long `if/elif` chain.


In [ ]:
MATERIALS = {
    ("junk", "a"):   {"name": "toothpicks",  "weight": 0.5, "strength": 0.3},
    ("junk", "b"):   {"name": "rotten_wood", "weight": 0.8, "strength": 0.2},
    ("common", "a"): {"name": "wood",        "weight": 1.0, "strength": 1.0},
    ("common", "b"): {"name": "brick",       "weight": 2.0, "strength": 1.6},
    ("strong", "a"): {"name": "steel",       "weight": 2.5, "strength": 4.5},
    ("strong", "b"): {"name": "concrete",    "weight": 3.0, "strength": 3.0},
}


def get_material(tier, material_digit):
    group = "a" if material_digit <= 4 else "b"
    return MATERIALS[(tier, group)]


def units_from_amount(amount):
    return amount * UNITS_PER_AMOUNT


print("Example lookup:", get_material("strong", 9))


## 3. Rolls — turn a 5-digit number into a floor recipe


In [ ]:
import random


def generate_roll():
    return random.randint(10000, 99999)


def full_decode(roll):
    text = str(roll)
    digit1 = int(text[0])
    digit2 = int(text[1])
    digit3 = int(text[2])
    digit4 = int(text[3])
    digit5 = int(text[4])

    tier = TIER_RANGES[digit1]
    amount = 10 if digit2 == 0 else digit2
    height = digit4 + digit5
    if height < 2:
        height = 2

    material = get_material(tier, digit3)
    units = units_from_amount(amount)

    return {
        "roll": roll,
        "tier": tier,
        "material_name": material["name"],
        "weight_per_unit": material["weight"],
        "capacity_mult": material["strength"],
        "units": units,
        "height": height,
    }


demo_roll = generate_roll()
print("Sample roll", demo_roll, "→", full_decode(demo_roll))


## 4. Linked-list node — one floor

Each floor points to the floor **above** and **below**.


In [ ]:
class Floor:
    def __init__(self, roll_data):
        self.material_name = roll_data["material_name"]
        self.tier = roll_data["tier"]
        self.height = roll_data["height"]
        self.units = roll_data["units"]
        self.weight_per_unit = roll_data["weight_per_unit"]
        self.strength = roll_data["capacity_mult"]
        self.health = 1.0
        self.above = None
        self.below = None

    def get_weight(self):
        return self.units * self.weight_per_unit

    def get_max_load(self):
        return (
            self.strength
            * self.height
            * CAPACITY_CONSTANT
            * self.health
        )

    def take_damage(self, percent):
        self.health = max(0.0, self.health * (1.0 - percent))

    def label(self):
        return f"{self.material_name}:{self.height}m"


## 5. Tower — linked list + recursion + stack

- **Linked list** for insert anywhere
- **Recursion** for weight above a floor
- **Stack** for collapse (remove broken floor + everything above)


In [ ]:
class Tower:
    def __init__(self):
        self.ground = None
        self.top = None
        self.size = 0
        self.height_m = 0.0

    def floor_count(self):
        return self.size

    def get_height(self):
        return self.height_m

    def floors_from_ground(self):
        result = []
        floor = self.ground
        while floor is not None:
            result.append(floor)
            floor = floor.above
        return result

    def insert_at(self, position, new_floor):
        if position < 0 or position > self.size:
            raise IndexError("bad position")

        if self.ground is None:
            self.ground = new_floor
            self.top = new_floor
            new_floor.above = None
            new_floor.below = None
        elif position == 0:
            new_floor.above = self.ground
            new_floor.below = None
            self.ground.below = new_floor
            self.ground = new_floor
        elif position == self.size:
            new_floor.below = self.top
            new_floor.above = None
            self.top.above = new_floor
            self.top = new_floor
        else:
            floor_below = self.ground
            for _ in range(position - 1):
                floor_below = floor_below.above
            floor_above = floor_below.above
            floor_below.above = new_floor
            new_floor.below = floor_below
            new_floor.above = floor_above
            floor_above.below = new_floor

        self.size += 1
        self.height_m += new_floor.height

    def remove(self, floor):
        if floor.below is not None:
            floor.below.above = floor.above
        else:
            self.ground = floor.above

        if floor.above is not None:
            floor.above.below = floor.below
        else:
            self.top = floor.below

        self.height_m -= floor.height
        self.size -= 1
        floor.above = None
        floor.below = None

    def load_above(self, floor):
        # recursion
        if floor.above is None:
            return 0.0
        next_floor = floor.above
        return next_floor.get_weight() + self.load_above(next_floor)

    def hold_ratio(self, floor):
        load = self.load_above(floor)
        if load == 0:
            return 999.0
        return floor.get_max_load() / load

    def find_weakest_floor(self):
        floors = self.floors_from_ground()
        if not floors:
            return None
        weakest = floors[0]
        worst = self.hold_ratio(weakest)
        for floor in floors[1:]:
            ratio = self.hold_ratio(floor)
            if ratio < worst:
                worst = ratio
                weakest = floor
        return weakest

    def check_stable(self):
        if self.ground is None:
            return 999.0, None
        weakest = self.find_weakest_floor()
        return self.hold_ratio(weakest), weakest

    def collapse(self, broken_floor=None):
        # stack
        if broken_floor is None:
            ratio, broken_floor = self.check_stable()
            if broken_floor is None or ratio >= 1.0:
                return [], 0.0

        if self.ground is None:
            return [], 0.0
        if broken_floor not in self.floors_from_ground():
            return [], 0.0

        stack = []
        floor = broken_floor
        while floor is not None:
            stack.append(floor)
            floor = floor.above

        fallen = []
        meters_lost = 0.0
        while stack:
            falling_floor = stack.pop()
            meters_lost += falling_floor.height
            self.remove(falling_floor)
            fallen.append(falling_floor)

        ratio, weakest = self.check_stable()
        if weakest is not None and ratio < 1.0:
            more_fallen, more_meters = self.collapse(weakest)
            fallen.extend(more_fallen)
            meters_lost += more_meters

        return fallen, meters_lost


# quick sanity check
_t = Tower()
_t.insert_at(0, Floor(full_decode(47382)))
print("Tower height after one wood floor:", _t.get_height(), "m")


## 6. Challenges — min-heap + binary search


In [ ]:
import heapq
from bisect import bisect_right

EVENT_NAMES = ("earthquake", "windstorm", "drunk_driver")
DAMAGE = {
    "earthquake": EARTHQUAKE_DAMAGE,
    "windstorm": WINDSTORM_DAMAGE,
    "drunk_driver": DRUNK_DRIVER_DAMAGE,
}


def make_challenge_heights():
    heights = []
    lowest = CHALLENGE_MIN_HEIGHT
    for left in range(NUM_CHALLENGES, 0, -1):
        highest = CHALLENGE_MAX_HEIGHT - CHALLENGE_MIN_GAP * (left - 1)
        if lowest > highest:
            break
        height = random.randint(lowest, highest)
        heights.append(height)
        lowest = height + CHALLENGE_MIN_GAP
    return heights


def make_heap(heights=None):
    if heights is None:
        heights = make_challenge_heights()
    heap = list(heights)
    heapq.heapify(heap)
    return heap


def height_milestones(tower):
    milestones = []
    total = 0.0
    for floor in tower.floors_from_ground():
        total += floor.height
        milestones.append(total)
    return milestones


def find_floor_at_height(tower, target_height):
    floors = tower.floors_from_ground()
    if not floors:
        return None
    milestones = height_milestones(tower)
    index = bisect_right(milestones, target_height - 1e-9)
    if index >= len(floors):
        index = len(floors) - 1
    return floors[index]


def apply_event(tower, event_name):
    floors = tower.floors_from_ground()
    if not floors:
        return event_name + ": empty"

    damage = DAMAGE[event_name]
    percent = int(damage * 100)

    if event_name == "earthquake":
        for floor in floors:
            floor.take_damage(damage)
        return f"Earthquake -{percent}% all"
    if event_name == "windstorm":
        for floor in floors[-3:]:
            floor.take_damage(damage)
        return f"Windstorm -{percent}% top"

    ground = find_floor_at_height(tower, 0.1) or floors[0]
    ground.take_damage(damage)
    return f"Drunk driver -{percent}% ground"


def run_challenges(tower, challenge_heap):
    messages = []
    while challenge_heap and challenge_heap[0] <= tower.get_height():
        height = heapq.heappop(challenge_heap)
        event_name = random.choice(EVENT_NAMES)
        messages.append(f"@{height}m {apply_event(tower, event_name)}")
    return messages


print("Sample challenge heights:", make_challenge_heights())


## 7. Game — turns (no GUI)


In [ ]:
class Player:
    def __init__(self, name):
        self.name = name
        self.tower = Tower()
        self.rolls_left = ROLLS_PER_TURN
        self.pending_roll = None
        self.challenge_heap = make_heap()
        self.won = False

    def next_challenge(self):
        if self.challenge_heap:
            return self.challenge_heap[0]
        return None


class Game:
    def __init__(self, name1="P1", name2="P2"):
        self.players = [Player(name1), Player(name2)]
        self.current_index = 0
        self.phase = "roll"
        self.winner = None
        self.message = f"{name1}'s turn — type r to roll"

    def current(self):
        return self.players[self.current_index]

    def show(self):
        lines = [self.message, ""]
        for player in self.players:
            parts = [floor.label() for floor in player.tower.floors_from_ground()]
            stack = " | ".join(parts) if parts else "empty"
            nxt = player.next_challenge()
            lines.append(
                f"  {player.name}: {player.tower.get_height():.0f}m  "
                f"[{stack}]  next⚡={nxt}"
            )
        lines.append("")
        lines.append(
            f"phase={self.phase}  rolls_left={self.current().rolls_left}  "
            f"pending={self.current().pending_roll['material_name'] if self.current().pending_roll else None}"
        )
        return "\n".join(lines)

    def roll(self):
        if self.phase != "roll":
            raise RuntimeError("Place or scrap first")
        player = self.current()
        if player.rolls_left <= 0:
            raise RuntimeError("No rolls left")
        recipe = full_decode(generate_roll())
        player.pending_roll = recipe
        player.rolls_left -= 1
        self.phase = "place"
        self.message = (
            f"Rolled {recipe['material_name']} {recipe['height']}m "
            f"({recipe['tier']}) — p place top · i N insert · s scrap"
        )
        return recipe

    def scrap(self):
        if self.phase != "place" or not self.current().pending_roll:
            raise RuntimeError("Nothing to scrap")
        self.current().pending_roll = None
        self.message = "Scrapped"
        self._after_action()

    def place(self, position=None):
        if self.phase != "place" or not self.current().pending_roll:
            raise RuntimeError("Roll first")
        player = self.current()
        recipe = player.pending_roll
        if position is None:
            position = player.tower.floor_count()
        position = max(0, min(position, player.tower.floor_count()))

        player.tower.insert_at(position, Floor(recipe))
        player.pending_roll = None

        events = run_challenges(player.tower, player.challenge_heap)
        ratio, weakest = player.tower.check_stable()
        collapsed = False
        if weakest is not None and ratio < 1.0:
            fallen, lost = player.tower.collapse(weakest)
            collapsed = True
            self.message = (
                f"Collapse! lost {len(fallen)} floors ({lost:.0f}m). "
                f"Tower now {player.tower.get_height():.0f}m"
            )
        else:
            self.message = f"Placed — tower {player.tower.get_height():.0f}m"
        if events:
            self.message += " | " + "; ".join(events)

        if player.tower.get_height() >= TARGET_HEIGHT:
            self.winner = player
            player.won = True
            self.phase = "game_over"
            self.message = f"{player.name} wins at {player.tower.get_height():.0f}m!"
            return

        self._after_action()

    def _after_action(self):
        player = self.current()
        if player.rolls_left > 0:
            self.phase = "roll"
            self.message += f" — {player.name} roll again (r)"
        else:
            self.current_index = 1 - self.current_index
            nxt = self.current()
            nxt.rolls_left = ROLLS_PER_TURN
            nxt.pending_roll = None
            self.phase = "roll"
            self.message += f" — {nxt.name}'s turn (r)"


print("Game classes ready")


## 8. Play

Run the next cell, then type commands in the prompt:

| Command | Meaning |
|---------|---------|
| `r` | roll |
| `p` | place on top |
| `i 0` | insert at ground (slot 0) |
| `i 2` | insert at slot 2 |
| `s` | scrap |
| `q` | quit |

If prompts are annoying (e.g. Run All), use the **auto demo** cell under this one instead.


In [ ]:
def play(name1="P1", name2="P2"):
    game = Game(name1, name2)
    print(game.show())
    print()
    print("Commands: r · p · i N · s · q")

    while game.phase != "game_over":
        try:
            raw = input(f"{game.current().name}> ").strip().lower()
        except EOFError:
            print("No input — stopping. Run the auto-demo cell instead.")
            break

        if not raw:
            continue
        try:
            if raw == "q":
                print("Quit.")
                break
            if raw == "r":
                game.roll()
            elif raw == "p":
                game.place()
            elif raw == "s":
                game.scrap()
            elif raw.startswith("i"):
                parts = raw.split()
                if len(parts) != 2:
                    print("Usage: i <slot>")
                    continue
                game.place(int(parts[1]))
            else:
                print("Unknown. Use r / p / i N / s / q")
                continue
        except Exception as err:
            print("!", err)
            continue

        print()
        print(game.show())
        print()

    if game.winner:
        print("Winner:", game.winner.name)
    return game


# Uncomment to play interactively:
# play()

print("Play function defined. Uncomment play() in this cell, or run the auto-demo below.")


## 9. Auto demo (no typing)

Places every roll on top for a few turns so you can see collapses and challenges.


In [ ]:
def auto_demo(seed=11, max_actions=40):
    random.seed(seed)
    game = Game("P1", "P2")
    print(game.show())
    print("-" * 60)

    for _ in range(max_actions):
        if game.phase == "game_over":
            break
        if game.phase == "roll":
            game.roll()
            print(game.message)
        game.place()  # always on top for the demo
        print(game.message)
        for player in game.players:
            parts = [floor.label() for floor in player.tower.floors_from_ground()]
            stack = " | ".join(parts) if parts else "empty"
            print(
                f"  {player.name}: {player.tower.get_height():.0f}m  [{stack}]"
            )
        print("-" * 60)

    if game.winner:
        print("Winner:", game.winner.name)
    else:
        print("Demo finished (no winner yet — raise max_actions or lower TARGET_HEIGHT).")
    return game


auto_demo()


## Optional: button UI (needs `ipywidgets`)

```bash
pip install ipywidgets
```

Then run the next cell for clickable Roll / Place / Scrap.


In [ ]:
def play_widgets(name1="P1", name2="P2"):
    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output
    except ImportError:
        print("Install ipywidgets first:  pip install ipywidgets")
        return None

    game = Game(name1, name2)
    out = widgets.Output()
    status = widgets.HTML()

    def refresh():
        status.value = "<pre style='font-size:14px'>" + game.show() + "</pre>"

    def do_roll(_btn):
        with out:
            try:
                game.roll()
                clear_output(wait=True)
                print(game.message)
            except Exception as err:
                print("!", err)
        refresh()

    def do_place(_btn):
        with out:
            try:
                game.place()
                clear_output(wait=True)
                print(game.message)
            except Exception as err:
                print("!", err)
        refresh()

    def do_scrap(_btn):
        with out:
            try:
                game.scrap()
                clear_output(wait=True)
                print(game.message)
            except Exception as err:
                print("!", err)
        refresh()

    roll_btn = widgets.Button(description="Roll", button_style="success")
    place_btn = widgets.Button(description="Place on top", button_style="primary")
    scrap_btn = widgets.Button(description="Scrap", button_style="danger")
    roll_btn.on_click(do_roll)
    place_btn.on_click(do_place)
    scrap_btn.on_click(do_scrap)

    refresh()
    display(widgets.HBox([roll_btn, place_btn, scrap_btn]), status, out)
    return game


# Uncomment to use buttons:
# play_widgets()

print("Widget UI ready. Uncomment play_widgets() after installing ipywidgets.")
